# Inspect dhoogla/nftoniotv2 on Kaggle

Diagnostic-only notebook — no training. Run this, then copy the full output back so `preprocess.py`'s `TARGET_COL` / `FLOW_COLS` / `DROP_COLS` can be matched to this dataset's actual schema.

Attach the `dhoogla/nftoniotv2` dataset as input before running.

In [ ]:
import os

DATASET_ROOT = "/kaggle/input/datasets/dhoogla/nftoniotv2"
if not os.path.exists(DATASET_ROOT):
    DATASET_ROOT = "/kaggle/input"

print(f"Searching under {DATASET_ROOT}\n")

all_files = []
for r, _dirs, files in os.walk(DATASET_ROOT):
    for fn in files:
        p = os.path.join(r, fn)
        all_files.append((p, os.path.getsize(p)))

all_files.sort(key=lambda x: -x[1])
print(f"Found {len(all_files)} file(s) total:\n")
for p, sz in all_files:
    print(f"{sz / 1e6:10.1f} MB  {p}")

csv_files = [p for p, _sz in all_files if p.lower().endswith((".csv", ".parquet", ".feather"))]
print(f"\n{len(csv_files)} tabular file(s) found (csv/parquet/feather).")

In [ ]:
import pandas as pd


def load_head(path, n=5):
    if path.lower().endswith(".csv"):
        return pd.read_csv(path, nrows=n)
    if path.lower().endswith(".parquet"):
        return pd.read_parquet(path).head(n)
    if path.lower().endswith(".feather"):
        return pd.read_feather(path).head(n)
    raise ValueError(path)


for path in csv_files:
    print("=" * 100)
    print(path)
    try:
        head = load_head(path, n=5)
    except Exception as e:
        print("  Could not read:", repr(e))
        continue
    print(f"  Columns ({len(head.columns)}):", list(head.columns))
    print("  dtypes:")
    print(head.dtypes.to_string())
    print("  First rows:")
    print(head.to_string())
    print()

## Row counts and candidate label columns

Line count via `wc -l` (cheap, no full load into memory). Then, for any column whose name looks like a label/attack/class column, print value counts from a sample.

In [ ]:
LABEL_HINTS = ("attack", "label", "class", "category", "type")
SAMPLE_ROWS = 200_000

for path in csv_files:
    print("=" * 100)
    print(path)

    if path.lower().endswith(".csv"):
        n_lines = int(os.popen(f'wc -l "{path}"').read().split()[0]) - 1
        print(f"  Row count (approx, via wc -l): {n_lines}")
        sample = pd.read_csv(path, nrows=SAMPLE_ROWS)
    elif path.lower().endswith(".parquet"):
        sample = pd.read_parquet(path)
        print(f"  Row count: {len(sample)}")
        sample = sample.head(SAMPLE_ROWS)
    else:
        sample = load_head(path, n=SAMPLE_ROWS)
        print(f"  Row count (sample only): {len(sample)}")

    candidate_cols = [c for c in sample.columns if any(h in c.lower() for h in LABEL_HINTS)]
    print(f"  Candidate label/attack columns: {candidate_cols}")
    for c in candidate_cols:
        print(f"  -- value_counts for '{c}' (sample of {len(sample)} rows) --")
        print(sample[c].value_counts().to_string())
    print()